# Fase 2: Limpieza y Transformación del Dataset de RR.HH.

El objetivo es obtener un dataset limpio, sin nulos, con tipos correctos y listo para la fase de análisis y modelado.

## 📌 Pasos que se realizarán:
1. Eliminación de columnas constantes y filas duplicadas.
2. Normalización de texto en `JobRole`.
3. Corrección de errores tipográficos en `MaritalStatus`.
4. Imputación de valores nulos (numéricos → mediana, categóricos → moda).
5. Conversión de tipos de datos (float64 → int64 donde corresponda).
6. Codificación de variables binarias (`Attrition`, `OverTime`).
7. Validación de calidad.
8. Guardado del dataset limpio.

---

In [1]:
# Importar librerías y configurar
import pandas as pd
import numpy as np

# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)

In [3]:
# Cargar datos original y crear copia de trabajo
df = pd.read_csv("hr.csv")  # Ajustar ruta si es necesario
df_clean = df.copy()

print(f"Dimensiones originales: {df.shape}")
print(f"Columnas originales: {df.columns.tolist()}")

Dimensiones originales: (1474, 35)
Columnas originales: ['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


## 1. Eliminación de columnas constantes y filas duplicadas
- **Columnas constantes**: `EmployeeCount`, `Over18`, `StandardHours` (siempre 1, 'Y', 80 respectivamente). No aportan variabilidad.
- **Filas duplicadas**: se detectaron 4 filas duplicadas exactas en la Fase 1.

In [4]:
# Eliminar columnas constantes
def drop_constant_columns(df):
    """Elimina columnas que tienen un solo valor único (constantes)."""
    cols_to_drop = [col for col in df.columns if df[col].nunique() == 1]
    return df.drop(columns=cols_to_drop)

df_clean = drop_constant_columns(df_clean)
print(f"Columnas después de eliminar constantes: {df_clean.shape[1]}")
print(f"Columnas eliminadas: {set(df.columns) - set(df_clean.columns)}")

Columnas después de eliminar constantes: 32
Columnas eliminadas: {'StandardHours', 'Over18', 'EmployeeCount'}


In [5]:
# Eliminar filas duplicadas
def drop_duplicate_rows(df):
    """Elimina filas duplicadas (mantiene la primera aparición)."""
    return df.drop_duplicates()

df_clean = drop_duplicate_rows(df_clean)
print(f"Filas después de eliminar duplicados: {df_clean.shape[0]} (originales: {df.shape[0]})")

Filas después de eliminar duplicados: 1470 (originales: 1474)


## 2. Normalización de texto en `JobRole`
El problema: valores como `' sALES eXECUTIVE '` (espacios, mayúsculas incorrectas).  
Solución: eliminar espacios y aplicar formato título.

In [6]:
# Limpiar columna JobRole
def fix_job_role(df):
    df["JobRole"] = df["JobRole"].str.strip().str.title()
    return df

df_clean = fix_job_role(df_clean)
print("Valores únicos de JobRole después de limpiar:")
print(df_clean["JobRole"].unique())

Valores únicos de JobRole después de limpiar:
<StringArray>
[          'Sales Executive',        'Research Scientist',
     'Laboratory Technician',    'Manufacturing Director',
 'Healthcare Representative',                   'Manager',
      'Sales Representative',         'Research Director',
           'Human Resources']
Length: 9, dtype: str


## 3. Corrección de errata en `MaritalStatus`
Se encontró el valor `'Marreid'` en lugar de `'Married'`. Lo corregimos.

In [7]:
# Corregir MaritalStatus
def fix_marital_status(df):
    df["MaritalStatus"] = df["MaritalStatus"].replace("Marreid", "Married")
    return df

df_clean = fix_marital_status(df_clean)
print("Valores únicos de MaritalStatus después de corrección:")
print(df_clean["MaritalStatus"].unique())

Valores únicos de MaritalStatus después de corrección:
<StringArray>
['Single', 'Married', 'Divorced', nan]
Length: 4, dtype: str


## 4. Imputación de valores nulos
- **Columnas numéricas**: `Age`, `JobSatisfaction`, `MonthlyIncome`, `TrainingTimesLastYear`, `YearsWithCurrManager` → se rellenan con la **mediana** (robusta frente a outliers).
- **Columnas categóricas**: `MaritalStatus`, `BusinessTravel`, `EducationField`, `OverTime`, `Department` → se rellenan con la **moda** (valor más frecuente).

In [8]:
# Imputar nulos en numéricas con la mediana
numeric_cols = ['Age', 'JobSatisfaction', 'MonthlyIncome', 'TrainingTimesLastYear', 'YearsWithCurrManager']
for col in numeric_cols:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f"{col}: nulos rellenados con mediana = {median_val}")

Age: nulos rellenados con mediana = 36.0
JobSatisfaction: nulos rellenados con mediana = 3.0
MonthlyIncome: nulos rellenados con mediana = 4907.0
TrainingTimesLastYear: nulos rellenados con mediana = 3.0
YearsWithCurrManager: nulos rellenados con mediana = 3.0


In [9]:
# Imputar nulos en categóricas con la moda
categorical_cols = ['MaritalStatus', 'BusinessTravel', 'EducationField', 'OverTime', 'Department']
for col in categorical_cols:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f"{col}: nulos rellenados con moda = '{mode_val}'")

MaritalStatus: nulos rellenados con moda = 'Married'
BusinessTravel: nulos rellenados con moda = 'Travel_Rarely'
EducationField: nulos rellenados con moda = 'Life Sciences'
OverTime: nulos rellenados con moda = 'No'
Department: nulos rellenados con moda = 'Research & Development'


## 5. Conversión de tipos de datos
Una vez imputados los nulos, convertimos las columnas numéricas que estaban como `float64` a `int64`.

In [10]:
# Convertir tipos a enteros
cols_to_int = ['Age', 'JobSatisfaction', 'MonthlyIncome', 'TrainingTimesLastYear', 'YearsWithCurrManager']
for col in cols_to_int:
    df_clean[col] = df_clean[col].astype(int)
    print(f"{col} ahora es {df_clean[col].dtype}")

Age ahora es int64
JobSatisfaction ahora es int64
MonthlyIncome ahora es int64
TrainingTimesLastYear ahora es int64
YearsWithCurrManager ahora es int64


## 6. Codificación de variables binarias
Las columnas `Attrition` y `OverTime` tienen valores `'Yes'`/`'No'`. Las transformamos a 1/0.
- `'Yes'` → 1
- `'No'` → 0

In [ ]:
# Codificar binarias
def encode_binary(df):
    mapping = {'Yes': 1, 'No': 0}
    df['Attrition'] = df['Attrition'].map(mapping)
    df['OverTime'] = df['OverTime'].map(mapping)
    return df

df_clean = encode_binary(df_clean)
print("Valores únicos en Attrition:", df_clean['Attrition'].unique())
print("Valores únicos en OverTime:", df_clean['OverTime'].unique())

## 7. Control de calidad (validaciones)
Verificamos que:
- No queden nulos en las columnas clave.
- Los tipos sean correctos.
- Las columnas binarias solo tengan 0 y 1.
- No exista el typo 'Marreid'.

In [ ]:
# Validaciones
assert df_clean[['Age', 'JobSatisfaction', 'MonthlyIncome', 'TrainingTimesLastYear', 'YearsWithCurrManager']].isnull().sum().sum() == 0, "❌ Aún hay nulos en columnas numéricas"
assert df_clean[['MaritalStatus', 'BusinessTravel', 'EducationField', 'OverTime', 'Department']].isnull().sum().sum() == 0, "❌ Aún hay nulos en columnas categóricas"
assert df_clean['Age'].dtype == 'int64', "❌ Age no es int64"
assert df_clean['Attrition'].dtype == 'int64', "❌ Attrition no es int64"
assert set(df_clean['Attrition'].unique()).issubset({0,1}), "❌ Attrition tiene valores fuera de {0,1}"
assert 'Marreid' not in df_clean['MaritalStatus'].unique(), "❌ Todavía existe 'Marreid'"

print("✅ ¡Todos los controles pasaron! El dataset está limpio.")

## 8. Guardar dataset limpio
Exportamos el resultado a un nuevo archivo CSV para usarlo en la Fase 3.

In [ ]:
# Guardar
df_clean.to_csv("hr_clean.csv", index=False)
print(f"Dataset limpio guardado como 'hr_clean.csv'")
print(f"Dimensiones finales: {df_clean.shape[0]} filas, {df_clean.shape[1]} columnas")
print("\nVista previa de las primeras 5 filas:")
df_clean.head()

## Resumen final de transformaciones aplicadas

| Acción                                      | Columnas afectadas                                                                 | Observación                                               |
|---------------------------------------------|------------------------------------------------------------------------------------|-----------------------------------------------------------|
| Eliminar columnas constantes                | `EmployeeCount`, `Over18`, `StandardHours`                                        | No aportan variabilidad                                   |
| Eliminar filas duplicadas                   | 4 filas duplicadas exactas                                                         | Quedan 1470 filas                                         |
| Limpiar `JobRole`                           | `JobRole`                                                                          | strip() + title()                                         |
| Corregir `MaritalStatus`                    | `MaritalStatus` (cambiar 'Marreid' → 'Married')                                    | Unificación de categorías                                 |
| Imputar nulos numéricos con mediana         | `Age`, `JobSatisfaction`, `MonthlyIncome`, `TrainingTimesLastYear`, `YearsWithCurrManager` | Robusto frente a outliers                                 |
| Imputar nulos categóricos con moda          | `MaritalStatus`, `BusinessTravel`, `EducationField`, `OverTime`, `Department`      | Valor más frecuente                                       |
| Convertir tipos a int                       | Las mismas numéricas anteriores                                                    | Ya no hay decimales innecesarios                          |
| Codificar binarias                          | `Attrition`, `OverTime` (Yes→1, No→0)                                              | Listas para modelado                                      |

✅ **El dataset resultante (`hr_clean.csv`) tiene 1470 filas y 31 columnas** (se eliminaron 4 columnas inútiles).